# Análisis de Movimientos de Cartera AFP

## 1. Importaciones

In [21]:
import pandas as pd

## 2. Configuración de Rutas

In [22]:
# Definir las rutas de los archivos
ruta_hist = r"Datos\Formato_351.csv"
ruta_enero = r"Datos\2026_1portainvdeta.xls"
ruta_febrero = r"Datos\2026_2portainvdeta.xls"
ruta_reporte = r"Datos\Reporte_Movimientos_AFP.xlsx"

print("Rutas configuradas:")
print(f"  - Histórico: {ruta_hist}")
print(f"  - Enero: {ruta_enero}")
print(f"  - Febrero: {ruta_febrero}")
print(f"  - Reporte: {ruta_reporte}")

Rutas configuradas:
  - Histórico: Datos\Formato_351.csv
  - Enero: Datos\2026_1portainvdeta.xls
  - Febrero: Datos\2026_2portainvdeta.xls
  - Reporte: Datos\Reporte_Movimientos_AFP.xlsx


## 3. Cargar Datos Históricos

In [23]:
# Columnas a leer del CSV histórico
cols_hist = [
    "Nombre de Entidad",
    "Fecha de Corte",
    "Nombre Patrimonio",
    "Razon_Social_Emisor",
    "Nemotecnico",
    "Codigo_Moneda",
    "Pais_Emisor",
    "Valor_Mercado_O_Pres_Pesos"
]

# Leer archivo CSV
df_hist = pd.read_csv(
    ruta_hist,
    usecols=cols_hist,
    encoding="latin1",
    low_memory=False
)

# Renombrar columna
df_hist = df_hist.rename(columns={
    "Valor_Mercado_O_Pres_Pesos": "Valor_Mercado"
})

print(f"Datos históricos cargados: {df_hist.shape[0]} filas")
print(f"\nPrimeras filas:")
df_hist.head()

Datos históricos cargados: 2290145 filas

Primeras filas:


,Nombre de Entidad,Fecha de Corte,Nombre Patrimonio,Razon_Social_Emisor,Nemotecnico,Codigo_Moneda,Valor_Mercado,Pais_Emisor
0,PROTECCION,31/01/2016,FONDO DE CESANTIAS,CARTERA COLECTIVA ABIERTA SUMAR,NaN,PESO,"$7,798,554.74",Colombia
1,PROTECCION,31/01/2016,FONDO DE CESANTIAS,CARTERA COLECTIVA ABIERTA CREDIFONDO,NaN,PESO,"$10,065,324.79",Colombia
2,PROTECCION,31/01/2016,FONDO DE CESANTIAS,CARTERA COLECTIVA ABIERTA OLIMPIA,NaN,PESO,"$7,453,464.58",Colombia
3,PROTECCION,31/01/2016,FONDO DE CESANTIAS,GOLDMAN SACHS JAPAN PORTFOLIO,H_GSJIUSH LX,USD,"$11,886,606,056.74",Luxemburgo
4,PROTECCION,31/01/2016,FONDO DE CESANTIAS,SPDR SP 500 ETF TRUST,Y_SPY,USD,"$12,078,951,397.18",Estados Unidos de AmÃ©rica


## 4. Definir Estructura y Función para Cargar Excel

In [24]:
# Columnas a leer de los archivos Excel
cols_excel = [
    "Nombre de Entidad",
    "Fecha de Corte",
    "Nombre Patrimonio",
    "Razon Social Emisor",
    "Nemotecnico",
    "Código Moneda",
    "Pais_Emisor",
    "Vr. mercado o Vr presente en $"
]

def cargar_excel(ruta):
    """Cargar y procesar archivo Excel"""
    df = pd.read_excel(
        ruta,
        sheet_name="Formato_351",
        usecols=cols_excel
    )

    df = df.rename(columns={
        "Razon Social Emisor": "Razon_Social_Emisor",
        "Código Moneda": "Codigo_Moneda",
        "Vr. mercado o Vr presente en $": "Valor_Mercado"
    })

    return df

print("Función cargar_excel definida")

Función cargar_excel definida


## 5. Cargar Datos de Enero y Febrero

In [25]:
# Cargar datos mensuales
df_enero = cargar_excel(ruta_enero)
df_febrero = cargar_excel(ruta_febrero)

print(f"Datos enero: {df_enero.shape[0]} filas")
print(f"Datos febrero: {df_febrero.shape[0]} filas")

Datos enero: 23552 filas
Datos febrero: 23714 filas


## 6. Combinar Todos los Datos

In [26]:
# Concatenar todos los dataframes
df = pd.concat([df_hist, df_enero, df_febrero], ignore_index=True)

# Convertir la columna a datetime para evitar problemas con tipos mixtos
df['Fecha de Corte'] = pd.to_datetime(df['Fecha de Corte'], errors='coerce')

print(f"Observaciones totales después de combinar: {df.shape[0]}")
print(f"\nRango de fechas: {df['Fecha de Corte'].min()} a {df['Fecha de Corte'].max()}")

C:\Users\gabri\AppData\Local\Temp\ipykernel_13644\1952507923.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Fecha de Corte'] = pd.to_datetime(df['Fecha de Corte'], errors='coerce')


Observaciones totales después de combinar: 2337411

Rango de fechas: 2016-01-31 00:00:00 a 2026-02-28 00:00:00


## 7. Limpieza de Datos

In [27]:
# Convertir fecha de corte a datetime
df["Fecha de Corte"] = pd.to_datetime(df["Fecha de Corte"], dayfirst=True)

# Limpiar valores de mercado: remover $ y comas
df["Valor_Mercado"] = (
    df["Valor_Mercado"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)

# Convertir a numérico
df["Valor_Mercado"] = pd.to_numeric(df["Valor_Mercado"], errors="coerce")

# Eliminar valores nulos en campos críticos
df = df.dropna(subset=["Nemotecnico", "Valor_Mercado"])

# Filtrar solo moneda USD
df = df[df["Codigo_Moneda"] == "USD"]

print(f"Observaciones en USD después de limpieza: {df.shape[0]}")
print(f"\nResumen de Valor_Mercado:")
print(df["Valor_Mercado"].describe())

Observaciones en USD después de limpieza: 15302

Resumen de Valor_Mercado:
count    1.530200e+04
mean     3.072171e+10
std      1.157866e+11
min      0.000000e+00
25%      3.085290e+09
50%      8.403798e+09
75%      2.025613e+10
max      3.523540e+12
Name: Valor_Mercado, dtype: float64


## 8. Construcción de Posiciones

In [28]:
# Agregar por posición (entidad, patrimonio, nemotecnico, fecha)
df = (
    df.groupby(
        [
            "Nombre de Entidad",
            "Nombre Patrimonio",
            "Nemotecnico",
            "Fecha de Corte"
        ]
    )["Valor_Mercado"]
    .sum()
    .reset_index()
)

# Ordenar datos
df = df.sort_values(
    [
        "Nombre de Entidad",
        "Nombre Patrimonio",
        "Nemotecnico",
        "Fecha de Corte"
    ]
)

print(f"Posiciones únicas: {df.shape[0]}")
print(f"\nMuestra de posiciones:")
df.head(10)

Posiciones únicas: 7649

Muestra de posiciones:


,Nombre de Entidad,Nombre Patrimonio,Nemotecnico,Fecha de Corte,Valor_Mercado
0,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE CESANTIAS LEY 50,BGLT30150645,2026-01-31,1.646624e+09
1,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE CESANTIAS LEY 50,BGLT30150645,2026-02-28,1.693027e+09
2,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE PENSIONES COLFONDOS CONSERVADOR,BGLT30150645,2026-01-31,7.958684e+09
3,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE PENSIONES COLFONDOS CONSERVADOR,BGLT30150645,2026-02-28,1.382639e+10
4,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE PENSIONES COLFONDOS ESEPCIAL DE RETIR...,BGLT30150645,2026-02-28,1.410856e+10
5,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE PENSIONES COLFONDOS MAYOR RIESGO,BGLT30150645,2026-01-31,3.238361e+10
6,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE PENSIONES COLFONDOS MAYOR RIESGO,BGLT30150645,2026-02-28,3.329620e+10
7,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE PENSIONES MODERADO,BGLT30150645,2026-01-31,8.850605e+10
8,"""COLFONDOS S.A."" Y ""COLFONDOS""",FONDO DE PENSIONES MODERADO,BGLT30150645,2026-02-28,9.100021e+10
9,"""PORVENIR""",FONDO DE CESANTIAS,BGLT10250435,2026-01-31,8.272065e+09


## 9. Calcular Pesos en el Portafolio

In [29]:
# Calcular valor total del portafolio por fecha y patrimonio
df["Valor_Portafolio"] = df.groupby(
    ["Nombre de Entidad", "Nombre Patrimonio", "Fecha de Corte"]
)["Valor_Mercado"].transform("sum")

# Calcular peso de cada posición
df["Peso"] = df["Valor_Mercado"] / df["Valor_Portafolio"]

print("Pesos calculados:")
print(df[["Nemotecnico", "Valor_Mercado", "Valor_Portafolio", "Peso"]].head(10))

Pesos calculados:
    Nemotecnico  Valor_Mercado  Valor_Portafolio      Peso
0  BGLT30150645   1.646624e+09      1.646624e+09  1.000000
1  BGLT30150645   1.693027e+09      1.693027e+09  1.000000
2  BGLT30150645   7.958684e+09      7.958684e+09  1.000000
3  BGLT30150645   1.382639e+10      1.382639e+10  1.000000
4  BGLT30150645   1.410856e+10      1.410856e+10  1.000000
5  BGLT30150645   3.238361e+10      3.238361e+10  1.000000
6  BGLT30150645   3.329620e+10      3.329620e+10  1.000000
7  BGLT30150645   8.850605e+10      8.850605e+10  1.000000
8  BGLT30150645   9.100021e+10      9.100021e+10  1.000000
9  BGLT10250435   8.272065e+09      2.261053e+11  0.036585


## 10. Análisis de Cambios en Posiciones

In [30]:
# Calcular cambios respecto al período anterior
df["Cambio_Peso"] = df.groupby(
    ["Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico"]
)["Peso"].diff()

df["Cambio_Valor"] = df.groupby(
    ["Nombre de Entidad", "Nombre Patrimonio", "Nemotecnico"]
)["Valor_Mercado"].diff()

# Clasificar como compra o venta
df["Tipo_Movimiento"] = df["Cambio_Valor"].apply(
    lambda x: "COMPRA" if x > 0 else "VENTA"
)

print("Cambios calculados:")
print(df[["Nemotecnico", "Cambio_Peso", "Cambio_Valor", "Tipo_Movimiento"]].head(10))

Cambios calculados:
    Nemotecnico  Cambio_Peso  Cambio_Valor Tipo_Movimiento
0  BGLT30150645          NaN           NaN           VENTA
1  BGLT30150645          0.0  4.640295e+07          COMPRA
2  BGLT30150645          NaN           NaN           VENTA
3  BGLT30150645          0.0  5.867705e+09          COMPRA
4  BGLT30150645          NaN           NaN           VENTA
5  BGLT30150645          NaN           NaN           VENTA
6  BGLT30150645          0.0  9.125913e+08          COMPRA
7  BGLT30150645          NaN           NaN           VENTA
8  BGLT30150645          0.0  2.494158e+09          COMPRA
9  BGLT10250435          NaN           NaN           VENTA


## 11. Filtrar Movimientos Significativos

In [31]:
# Definir umbral minimo para cambios de peso
umbral = 0.01  # 1%

# Filtrar movimientos
movimientos = df[df["Cambio_Peso"].abs() > umbral].copy()

# Ordenar por cambio de peso (descendente en valor absoluto)
movimientos = movimientos.sort_values(
    "Cambio_Peso",
    key=abs,
    ascending=False
)

print(f"Movimientos significativos (umbral > {umbral*100}%): {movimientos.shape[0]}")
print(f"\nMovimientos por tipo:")
print(movimientos["Tipo_Movimiento"].value_counts())

Movimientos significativos (umbral > 1.0%): 1457

Movimientos por tipo:
Tipo_Movimiento
COMPRA    764
VENTA     693
Name: count, dtype: int64


## 12. Segmentar por Tipo de Movimiento

In [32]:
# Crear reporte con columnas finales
reporte = movimientos[
    [
        "Nombre de Entidad",
        "Nombre Patrimonio",
        "Nemotecnico",
        "Fecha de Corte",
        "Valor_Mercado",
        "Valor_Portafolio",
        "Peso",
        "Cambio_Peso",
        "Cambio_Valor",
        "Tipo_Movimiento"
    ]
]

# Separar compras y ventas
compras = reporte[reporte["Tipo_Movimiento"] == "COMPRA"]
ventas = reporte[reporte["Tipo_Movimiento"] == "VENTA"]

print(f"Total de movimientos: {reporte.shape[0]}")
print(f"  - Compras: {compras.shape[0]}")
print(f"  - Ventas: {ventas.shape[0]}")
print(f"\nMuestra de compras:")
compras.head()

Total de movimientos: 1457
  - Compras: 764
  - Ventas: 693

Muestra de compras:


,Nombre de Entidad,Nombre Patrimonio,Nemotecnico,Fecha de Corte,Valor_Mercado,Valor_Portafolio,Peso,Cambio_Peso,Cambio_Valor,Tipo_Movimiento
2588,PORVENIR,FONDO DE PENSIONES OBLIGATORIO MODERADO,BANCOG070827,2019-08-31,1.795797e+10,1.795797e+10,1.000000,0.990894,6.991164e+08,COMPRA
617,PORVENIR,FONDO DE CESANTIAS,AVH US,2019-01-31,5.493602e+08,2.382200e+10,0.023061,-0.976939,5.292841e+07,COMPRA
6411,PROTECCION,FONDO DE PENSIONES OBLIGATORIAS PROTECCIÃN RE...,US279158AL39,2019-08-31,2.722934e+10,2.722934e+10,1.000000,0.968257,4.031529e+09,COMPRA
4279,PROTECCION,FONDO DE PENSIONES MODERADO,US279158AK55,2019-08-31,1.810620e+10,1.993171e+10,0.908412,0.907277,2.646583e+09,COMPRA
1497,PORVENIR,FONDO DE PENSIONES OBLIGATORIAS PORVENIR CONSE...,YUSD15052049,2020-01-31,2.163282e+09,2.163282e+09,1.000000,0.862993,1.783350e+08,COMPRA


## 13. Exportar Reporte en Excel

In [33]:
# Crear archivo Excel con múltiples hojas
with pd.ExcelWriter(ruta_reporte, engine="xlsxwriter") as writer:

    # Escribir datos en hojas
    reporte.to_excel(writer, sheet_name="Movimientos", index=False)
    compras.to_excel(writer, sheet_name="Compras", index=False)
    ventas.to_excel(writer, sheet_name="Ventas", index=False)

    workbook = writer.book

    # Definir formatos
    header = workbook.add_format({
        "bold": True,
        "align": "center",
        "border": 1,
        "bg_color": "#E7E6E6"
    })

    money = workbook.add_format({
        "num_format": "$#,##0",
        "border": 1
    })

    percent = workbook.add_format({
        "num_format": "0.00%",
        "border": 1
    })

    text = workbook.add_format({"border": 1})

    # Función para aplicar formato
    def formato(nombre, df):
        ws = writer.sheets[nombre]

        # Encabezados
        for i, col in enumerate(df.columns):
            ws.write(0, i, col, header)

        # Ancho de columnas y formato
        ws.set_column("A:A", 22, text)
        ws.set_column("B:B", 30, text)
        ws.set_column("C:C", 12, text)
        ws.set_column("D:D", 14, text)
        ws.set_column("E:E", 18, money)
        ws.set_column("F:F", 18, money)
        ws.set_column("G:G", 12, percent)
        ws.set_column("H:H", 12, percent)
        ws.set_column("I:I", 18, money)
        ws.set_column("J:J", 14, text)

        # Congelar encabezado
        ws.freeze_panes(1, 0)
        # Agregar filtros
        ws.autofilter(0, 0, len(df), len(df.columns) - 1)

    # Aplicar formato a todas las hojas
    formato("Movimientos", reporte)
    formato("Compras", compras)
    formato("Ventas", ventas)

print("Reporte generado exitosamente.")
print(f"Ubicación: {ruta_reporte}")

Reporte generado exitosamente.
Ubicación: Datos\Reporte_Movimientos_AFP.xlsx
